# Explore the DDI Knowledge Graph

This notebook explores the heterogeneous Drug-Drug Interaction (DDI) knowledge graph built in `scripts/02_build_kg.py`.

In [ ]:
import torch
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load the built HeteroData object
graph_path = Path("../data/graph/hetero_data.pt")
if graph_path.exists():
    data = torch.load(graph_path)
    print("=== HeteroData Graph Loaded Successfully ===")
    print(data)
else:
    # Fallback to simulated graph structure for demo purposes
    from torch_geometric.data import HeteroData
    data = HeteroData()
    data['drug'].num_nodes = 100
    data['protein'].num_nodes = 500
    data['side_effect'].num_nodes = 200
    data['drug', 'binds', 'protein'].edge_index = torch.randint(0, 100, (2, 1200))
    data['protein', 'interacts', 'protein'].edge_index = torch.randint(0, 500, (2, 3500))
    data['drug', 'ddi', 'drug'].edge_index = torch.randint(0, 100, (2, 800))
    data['drug', 'ddi', 'drug'].side_effect_label = torch.rand(800, 200) > 0.98
    print("=== Using Simulated HeteroData Graph (Original not found) ===")
    print(data)

## 1. Graph Summary Statistics

In [ ]:
print(f"Drug nodes: {data['drug'].num_nodes}")
print(f"Protein nodes: {data['protein'].num_nodes}")
print(f"Side effect nodes: {data['side_effect'].num_nodes}")
print(f"Drug-protein interactions (bipartite): {data['drug', 'binds', 'protein'].edge_index.shape[1]}")
print(f"Protein-protein interactions (homophily): {data['protein', 'interacts', 'protein'].edge_index.shape[1]}")
print(f"Drug-drug prediction pairs: {data['drug', 'ddi', 'drug'].edge_index.shape[1]}")

## 2. Degree Distribution Analysis

In [ ]:
# Calculate drug degrees (drug-protein interactions)
dp_edges = data['drug', 'binds', 'protein'].edge_index.numpy()
drug_degrees = np.bincount(dp_edges[0], minlength=data['drug'].num_nodes)

plt.figure(figsize=(10, 5))
plt.hist(drug_degrees, bins=20, color='#4CAF50', edgecolor='black', alpha=0.7)
plt.title("Drug Degree Distribution (Drug-Protein Interactions)")
plt.xlabel("Degree (number of target proteins)")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Visualizing a local drug-protein subgraph

In [ ]:
# Sample first 5 drugs and their protein targets
G = nx.Graph()
for drug_idx in range(5):
    targets = dp_edges[1][dp_edges[0] == drug_idx]
    G.add_node(f"Drug_{drug_idx}", color='red')
    for t in targets[:5]:  # limit to 5 targets per drug for clarity
        G.add_node(f"Protein_{t}", color='blue')
        G.add_edge(f"Drug_{drug_idx}", f"Protein_{t}")

plt.figure(figsize=(12, 8))
colors = [nx.get_node_attributes(G, 'color')[node] for node in G.nodes()]
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=600, edgecolors='black')
nx.draw_networkx_edges(G, pos, alpha=0.5, width=1.5)
nx.draw_networkx_labels(G, pos, font_size=8, font_family='sans-serif')
plt.title("Bipartite Drug-Target Association Subgraph")
plt.axis('off')
plt.show()